In [ ]:
from transformers import AutoModelForCausalLM, GemmaConfig, AutoTokenizer, AutoModel, MistralConfig, MistralModel, MistralForCausalLM, LlamaConfig, LlamaForCausalLM
from transformers import Trainer, DataCollatorForLanguageModeling, TrainingArguments, TrainerCallback
import torch
import torch.nn as nn
import torch.nn.init as init
import json
import pickle
import pandas as pd
from datasets import Dataset
import math
import wandb
import ast

## Tokenizer

In [ ]:
token_path = "/kaggle/input/sangrah_tokenizer/transformers/default/1/sangrah_tokenizers/sangrah.csv_SentencePieceBPETokenizer_50000_transformer" ## CHANGE TOKENIZER PATH HERE
tokenizer = AutoTokenizer.from_pretrained(token_path)
# len(tokenizer.vocab), tokenizer

## LlamaConfig

In [ ]:
## CHNAGE MODEL CONFIGURATION HERE
config = LlamaConfig(hidden_size=512,
                     vocab_size=len(tokenizer.vocab),
                     num_attention_heads=8,
                     num_key_value_heads=2,
                     num_hidden_layers=16,
                     intermediate_size=1024)
config

LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 8,
  "num_hidden_layers": 16,
  "num_key_value_heads": 2,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "transformers_version": "4.45.1",
  "use_cache": true,
  "vocab_size": 49152
}

In [ ]:
## MODEL OVERVIEW
model = LlamaForCausalLM(config)
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 512)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=512, out_features=512, bias=False)
          (k_proj): Linear(in_features=512, out_features=128, bias=False)
          (v_proj): Linear(in_features=512, out_features=128, bias=False)
          (o_proj): Linear(in_features=512, out_features=512, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=512, out_features=1024, bias=False)
          (up_proj): Linear(in_features=512, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=512, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((512,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((512,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm

In [ ]:
## TOTAL PARAMS
total_param=0
for i,j in model.named_parameters():
    total_param += j.numel()
print(total_param/(10**6))

86.000128


## Dataset Loading

In [ ]:
dataset = pd.read_csv("/kaggle/input/tokenized-sangrah/kaggle/working/tokenized_data.csv") ## CHANGE THE PATH TO TOKENIZED DATASET HERE
# dataset = dataset.drop(columns = 'Unnamed: 0')                                                      ## REMOVED A COLUM THAT WAS UNNECESSARY
dataset['input_ids'] = dataset['input_ids'].apply(ast.literal_eval)                                 ## CONVERTED THE Str(List[Int]) to List[Int] (IMPORTANT)
display(dataset)                                                                                    

tokenized_data = Dataset.from_pandas(dataset)                                                       ## CONVERTED TO A DATASET OBJECT

print(tokenized_data)

,input_ids
0,"[5, 20632, 11371, 214, 569, 6748, 438, 11307, ..."
1,"[133, 1778, 405, 5839, 1007, 28641, 454, 235, ..."
2,"[132, 23605, 875, 8153, 566, 4104, 10071, 2003..."
3,"[22, 405, 1241, 6, 132, 5, 438, 1830, 224, 181..."
4,"[29497, 454, 5991, 599, 7045, 315, 6, 132, 5, ..."
...,...
33018,"[21248, 33, 158, 1591, 45741, 4591, 487, 75, 9..."
33019,"[189, 41472, 33, 489, 4366, 680, 9128, 31661, ..."
33020,"[489, 21046, 250, 9278, 3100, 9278, 459, 815, ..."
33021,"[25, 192, 241, 663, 284, 7565, 132, 486, 31275..."


Dataset({
    features: ['input_ids'],
    num_rows: 33023
})


In [ ]:
train_test_split = tokenized_data.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

# Data Collator for Language Modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # For causal language modeling
)
print(train_dataset, eval_dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 29720
}) Dataset({
    features: ['input_ids'],
    num_rows: 3303
})


## Training

In [ ]:
## WANDB = Weights and Biases (website) that tracks the training process, stores necesary information like loss and also the trained models. You will have to create an account on Weights and Biases website.
## Once account is created, only then run this cell and paste your API key in the text prompt. Your API key is available at https://wandb.ai/authorize or in your home page when you open Weights and Biases account.
wandb.init(project="nlp_nepali_training", name="sangrah") ## CHANGE AS PER YOUR WISH

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
## NO NEED TO CHANGE ANYTHING HERE
training_args = TrainingArguments(
    output_dir="./results",                # Set a local path to store results (checkpoints)
    eval_strategy="steps",           # Evaluate after every epoch
    logging_dir="./logs",                  # Directory for logs
    logging_steps=50,                      # Log every 50 steps
    save_steps=1000,                       # Save checkpoints every 1000 steps
    eval_steps = 1000,
    save_total_limit=3,                    # Limit the number of saved checkpoints
    per_device_train_batch_size=8,         # Adjust batch size based on your memory
    per_device_eval_batch_size=8,
    num_train_epochs=20,                    # Number of epochs
    report_to="wandb",                     # Log to W&B
    logging_first_step=True,
    load_best_model_at_end=True,           # Load the best model based on evaluation
    save_strategy="steps",                 # Save model after every epoch (or steps)
    push_to_hub=False,                       # Avoid pushing to hub (unless needed)
    fp16 = True
)

class PerplexityCallback(TrainerCallback):
    def __init__(self):
        self.perplexity_by_epoch = []

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            perplexity = math.exp(metrics["eval_loss"])
            self.perplexity_by_epoch.append({"epoch": state.epoch, "perplexity": perplexity})
            wandb.log({"eval_perplexity": perplexity})
            
    def on_log(self, args, state, control, logs=None, **kwargs):
        if "eval_loss" in logs:
            perplexity = math.exp(logs["eval_loss"])
            logs["eval_perplexity"] = perplexity
            wandb.log({"eval_perplexity": perplexity})

class ModelSaveCallback(TrainerCallback):
    def __init__(self, save_interval=0.25):
        super().__init__()
        self.save_interval = save_interval  # Interval to save model (fraction of epoch)
        self.last_saved_step = 0

    def on_step_end(self, args, state, control, **kwargs):
        # Calculate current progress in epochs
        current_epoch_fraction = state.global_step / state.max_steps * args.num_train_epochs
        
        # Check if it's time to save the model
        if (current_epoch_fraction - self.last_saved_step) >= self.save_interval:
            self.last_saved_step = current_epoch_fraction  # Update the last saved step
            
            # Save the model
            model_save_path = f"{wandb.run.dir}/model_epoch_{current_epoch_fraction:.2f}"
            model.save_pretrained(model_save_path)
            
            # Upload the saved model to W&B
            wandb.save(f"{model_save_path}/*")  # Save all files in the directory to W&B
            print(f"Model saved at {current_epoch_fraction:.2f} epochs")

perplexity_callback = PerplexityCallback()

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[PerplexityCallback(), ModelSaveCallback(save_interval=0.2)] 
)

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
wandb.watch(model, log="all", log_freq=100)
trainer.train()
wandb.finish()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss,Perplexity
1000,7.331900,7.272457,1440.084574
2000,6.678000,6.688414,803.047686
3000,6.340800,6.323770,557.671216
4000,6.092100,6.065169,430.595586
5000,5.853800,5.867444,353.344690
6000,5.569400,5.713341,302.881378
7000,5.486000,5.590077,267.756216
8000,5.373100,5.495382,243.564509


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(